In [ ]:
import numpy as np
import torch
import random

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/home/duarte/Desktop/Tese/Mapping_Tese/mapping_tese")
BIDS_ROOT = PROJECT_ROOT / "data/BIDS-somatosensory/BIDS-somatosensory"
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
EVENTS_DIR = BIDS_ROOT / "events"

RESULTS_BASE_DIR = PROJECT_ROOT / "notebooks/GNN/results/GCN_4Classes_MultiSubject"
RESULTS_BASE_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_EDGES_PATH = PROJECT_ROOT / "notebooks/GNN/results/graph_edges.pt"

session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs_per_subject = 4

HRF_DELAY = 6.0
WINDOW = 1

BATCH_SIZE = 16
MAX_EPOCHS = 1000
PATIENCE = 300
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.25
HIDDEN_DIM = 64

In [ ]:
import re

REGION_TO_ELECTRODES = {
    "Middle_Finger": ["E1", "E2", "E3"],
    "Hand": ["E4", "E5", "E6", "E7"],
    "Forearm": ["E8", "E9", "E10", "E11", "E12", "E13"],
    "Arm": ["E14", "E15", "E16", "E17", "E18", "E19", "E20"],
}
ELECTRODE_TO_REGION = {
    elec: region
    for region, elecs in REGION_TO_ELECTRODES.items()
    for elec in elecs
}
REGION_TO_LABEL = {"Middle_Finger": 0, "Hand": 1, "Forearm": 2, "Arm": 3}
LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

derivative_subjects = sorted(
    d.name for d in DERIVATIVES.iterdir()
    if d.is_dir() and d.name.startswith("sub-")
)

event_subjects = set()
for ev_path in EVENTS_DIR.glob(f"sub-*_{session}_{task}_run-*_events.tsv"):
    m = re.match(r"^(sub-[^_]+)_", ev_path.name)
    if m:
        event_subjects.add(m.group(1))
event_subjects = sorted(event_subjects)

subjects = sorted(set(derivative_subjects) & set(event_subjects))

print("Subjects in derivatives:", len(derivative_subjects))
print("Subjects in events:", len(event_subjects))
print("Subjects included:", len(subjects))
print(subjects)

In [ ]:
import json

def resolve_bold_path(subject, run, space_name):
    func_dir = DERIVATIVES / subject / session / "func"
    candidates = [
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space_name}_desc-preproc_bold.nii.gz",
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space_name}_desc-preproc_bold.nii",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.nii.gz",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.nii",
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

def get_tr(subject, run=1, default_tr=2.0):
    func_dir = DERIVATIVES / subject / session / "func"
    json_candidates = [
        func_dir / f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.json",
        func_dir / f"{subject}_{session}_{task}_run-{run}_desc-preproc_bold.json",
    ]
    for p in json_candidates:
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                meta = json.load(f)
            if "RepetitionTime" in meta and meta["RepetitionTime"] is not None:
                return float(meta["RepetitionTime"])
    return float(default_tr)

tr_by_subject = {s: get_tr(s, run=1, default_tr=2.0) for s in subjects}
print("TR values:", sorted(set(tr_by_subject.values())))

In [ ]:
import pandas as pd

def load_subject_events(subject):
    all_events = []
    required_cols = {"onset", "duration", "trial_type"}

    for run in range(1, n_runs_per_subject + 1):
        ev_path = EVENTS_DIR / f"{subject}_{session}_{task}_run-{run}_events.tsv"
        if not ev_path.exists():
            raise FileNotFoundError(f"Missing events file: {ev_path}")

        df = pd.read_csv(ev_path, sep="\t")
        df.columns = [str(c).strip().lower() for c in df.columns]

        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"{ev_path.name} missing columns {sorted(missing)}")

        df["run"] = run
        all_events.append(df)

    events_df = pd.concat(all_events, ignore_index=True)
    stim_events = events_df[~events_df["trial_type"].isin(["Baseline", "Jitter"])].copy()
    stim_events["region"] = stim_events["trial_type"].map(ELECTRODE_TO_REGION)

    unmapped = stim_events["region"].isna().sum()
    if unmapped > 0:
        bad = sorted(stim_events.loc[stim_events["region"].isna(), "trial_type"].unique())
        raise ValueError(f"Unmapped trial_type values for {subject}: {bad}")

    stim_events["label"] = stim_events["region"].map(REGION_TO_LABEL).astype(int)
    return events_df, stim_events

_preview_e, _preview_s = load_subject_events(subjects[0])
print("Preview subject:", subjects[0])
print("Stim samples:", len(_preview_s))
print("Class counts:", _preview_s["region"].value_counts().to_dict())

In [ ]:
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.image import load_img, new_img_like

atlas = fetch_atlas_destrieux_2009()
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()

s1_indices = [
    i for i, lab in enumerate(atlas.labels)
    if "L G_postcentral" in str(lab) and i != 0
]
mask_data = np.isin(atlas_data, s1_indices).astype("uint8")
s1_mask = new_img_like(atlas_img, mask_data)

print("Selected atlas indices:", len(s1_indices))

In [ ]:
def build_6nn_edge_index(voxel_coords):
    coord_to_idx = {tuple(c): i for i, c in enumerate(voxel_coords)}
    neigh = [(1,0,0),(-1,0,0),(0,1,0),(0,-1,0),(0,0,1),(0,0,-1)]

    edges = []
    for i, (x, y, z) in enumerate(voxel_coords):
        for dx, dy, dz in neigh:
            j = coord_to_idx.get((x+dx, y+dy, z+dz), None)
            if j is not None:
                edges.append([i, j])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index

In [ ]:
from nilearn.image import index_img, resample_to_img
from nilearn.maskers import NiftiMasker

first_subject = subjects[0]
first_run = resolve_bold_path(first_subject, run=1, space_name=space)
if first_run is None:
    raise FileNotFoundError(f"No first run found for {first_subject}")

first_img = load_img(str(first_run))
ref_img = index_img(first_img, 0)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation="nearest")
masker_ref = NiftiMasker(mask_img=s1_mask_resampled, standardize=None).fit(first_img)

mask_bool = masker_ref.mask_img_.get_fdata().astype(bool)
voxel_coords = np.column_stack(np.where(mask_bool))
n_voxels = voxel_coords.shape[0]

In [ ]:
if GRAPH_EDGES_PATH.exists():
    edge_index = torch.load(GRAPH_EDGES_PATH, map_location="cpu")
    print("Loaded edge_index from disk:", GRAPH_EDGES_PATH)
else:
    edge_index = build_6nn_edge_index(voxel_coords)
    torch.save(edge_index, GRAPH_EDGES_PATH)
    print("Built and saved edge_index:", GRAPH_EDGES_PATH)

In [ ]:
print("n_voxels:", n_voxels)
print("edge_index shape:", tuple(edge_index.shape))

In [ ]:
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn as nn
import torch.nn.functional as F

class SomatotopicGCN(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=64, n_classes=4, dropout=0.25):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.res_proj = nn.Linear(hidden_channels, hidden_channels)
        self.classifier = nn.Linear(hidden_channels, n_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        residual = self.res_proj(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        x = x + residual
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = global_mean_pool(x, batch)
        logits = self.classifier(x)
        return logits

In [ ]:
from nilearn.image import mean_img

#per subject feature extration
def extract_subject_samples(subject):
    tr_subject = tr_by_subject[subject]
    _, stim_events = load_subject_events(subject)

    run1_path = resolve_bold_path(subject, run=1, space_name=space)
    if run1_path is None:
        raise FileNotFoundError(f"Missing run-1 bold for {subject}")

    run1_img = load_img(str(run1_path))
    ref = index_img(run1_img, 0)
    local_mask = resample_to_img(s1_mask, ref, interpolation="nearest")
    masker = NiftiMasker(mask_img=local_mask, standardize=None).fit(run1_img)

    X_list, y_list, run_list = [], [], []

    for run in range(1, n_runs_per_subject + 1):
        bold_path = resolve_bold_path(subject, run=run, space_name=space)
        if bold_path is None:
            continue

        img = load_img(str(bold_path))
        run_len = img.shape[3]
        run_events = stim_events[stim_events["run"] == run].sort_values("onset")

        for _, ev in run_events.iterrows():
            center = int(np.round((float(ev["onset"]) + HRF_DELAY) / tr_subject))
            vols = list(range(max(0, center - WINDOW), min(run_len, center + WINDOW + 1)))
            if len(vols) == 0:
                continue

            avg_img = mean_img(index_img(img, vols), copy_header=True)
            feat = masker.transform(avg_img).ravel()

            if feat.shape[0] != n_voxels:
                raise ValueError(f"Voxel mismatch for {subject}, run {run}: {feat.shape[0]} vs {n_voxels}")

            X_list.append(feat.astype(np.float32))
            y_list.append(int(ev["label"]))
            run_list.append(int(run))

    if len(X_list) == 0:
        raise RuntimeError(f"No usable samples for {subject}")

    X = np.vstack(X_list)
    y = np.asarray(y_list, dtype=np.int64)
    groups = np.asarray(run_list, dtype=np.int64)

    return X, y, groups, tr_subject, masker

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*torch-scatter.*")

In [ ]:
from nilearn import datasets as nl_datasets, plotting
from nilearn import surface as surf_utils
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

fsaverage = nl_datasets.fetch_surf_fsaverage()

In [ ]:
import shap
import pickle
import torch.nn as nn

SHAP_BACKGROUND_SIZE = 32
SHAP_NSAMPLES = 50
shap_rng = np.random.default_rng(RANDOM_SEED)

class GCNShapWrapper(nn.Module):
    def __init__(self, base_model, edge_index, n_voxels):
        super().__init__()
        self.base_model = base_model
        self.n_voxels = n_voxels
        self.register_buffer("edge_index_single", edge_index)

    def _batched_edge_index(self, batch_size, device):
        ei = self.edge_index_single.to(device)
        offsets = (torch.arange(batch_size, device=device) * self.n_voxels).view(-1, 1, 1)
        ei_rep = ei.unsqueeze(0).repeat(batch_size, 1, 1) + offsets
        return ei_rep.permute(1, 0, 2).reshape(2, -1)

    def forward(self, x_flat):
        batch_size = x_flat.shape[0]
        device = x_flat.device
        x_node = x_flat.reshape(-1, 1)
        edge_index_batched = self._batched_edge_index(batch_size, device)
        batch_vec = torch.arange(batch_size, device=device).repeat_interleave(self.n_voxels)
        return self.base_model(x_node, edge_index_batched, batch_vec)

In [ ]:
def run_shap_for_subject(subject):
    subj_dir  = RESULTS_BASE_DIR / subject
    model_dir = subj_dir / "models"
    shap_dir  = subj_dir / "shap"
    shap_dir.mkdir(parents=True, exist_ok=True)

    try:
        X, y, groups, _, subj_masker = extract_subject_samples(subject)
    except Exception as exc:
        print(f"Skipping {subject}: {exc}")
        return None

    all_fold_shap, all_fold_labels = [], []

    for test_run in range(1, n_runs_per_subject + 1):
        model_path = model_dir / f"fold_{test_run}_model.pt"
        scaler_path = model_dir / f"fold_{test_run}_scaler.pkl"
        if not model_path.exists() or not scaler_path.exists():
            print(f"{subject} fold {test_run}: missing checkpoint, skipping "
                  f"(retrain with the updated training cell first)")
            continue

        val_run = (test_run % n_runs_per_subject) + 1
        train_runs = [r for r in range(1, n_runs_per_subject + 1) if r != test_run and r != val_run]

        train_mask = np.isin(groups, train_runs)
        test_mask = (groups == test_run)
        if not np.any(train_mask) or not np.any(test_mask):
            continue

        with open(scaler_path, "rb") as f:
            scaler = pickle.load(f)

        X_train_scaled = scaler.transform(X[train_mask]).astype(np.float32)
        X_test_scaled = scaler.transform(X[test_mask]).astype(np.float32)
        y_test_fold = y[test_mask]

        base_model = SomatotopicGCN(
            in_channels=1,
            hidden_channels=HIDDEN_DIM,
            n_classes=4,
            dropout=DROPOUT,
        ).to(device)
        base_model.load_state_dict(torch.load(model_path, map_location=device))
        base_model.eval()

        wrapped_model = GCNShapWrapper(base_model, edge_index, n_voxels).to(device)
        wrapped_model.eval()

        bg_size = min(SHAP_BACKGROUND_SIZE, len(X_train_scaled))
        bg_idx = shap_rng.choice(len(X_train_scaled), size=bg_size, replace=False)
        background = torch.from_numpy(X_train_scaled[bg_idx]).to(device)
        test_tensor = torch.from_numpy(X_test_scaled).to(device)

        explainer = shap.GradientExplainer(wrapped_model, background)
        raw_shap_values = explainer.shap_values(test_tensor, nsamples=SHAP_NSAMPLES)

        if isinstance(raw_shap_values, list):
            fold_shap = np.stack(raw_shap_values, axis=-1)
        else:
            fold_shap = np.asarray(raw_shap_values)

        if fold_shap.ndim != 3:
            raise ValueError(f"Unexpected SHAP shape for {subject} fold {test_run}: {fold_shap.shape}")
        if fold_shap.shape[1] != X_test_scaled.shape[1]:
            if fold_shap.shape[0] == 4 and fold_shap.shape[1] == len(X_test_scaled):
                fold_shap = np.transpose(fold_shap, (1, 2, 0))
            else:
                raise ValueError(f"SHAP shape {fold_shap.shape} does not match X shape {X_test_scaled.shape}")

        print(f"{subject} fold {test_run}: explained {fold_shap.shape[0]} held-out trials")

        all_fold_shap.append(fold_shap)
        all_fold_labels.append(y_test_fold)

    if not all_fold_shap:
        print(f"No SHAP results for {subject} (no fold checkpoints found)")
        return None

    shap_values = np.concatenate(all_fold_shap, axis=0)
    y_used = np.concatenate(all_fold_labels, axis=0)

    shap_abs = np.zeros((4, n_voxels), dtype=np.float32)
    shap_signed = np.zeros((4, n_voxels), dtype=np.float32)
    for c in range(4):
        class_mask = y_used == c
        if not np.any(class_mask):
            continue
        class_shap = shap_values[class_mask, :, c]
        shap_abs[c] = np.mean(np.abs(class_shap), axis=0)
        shap_signed[c] = np.mean(class_shap, axis=0)

    np.save(shap_dir / "shap_values_all_folds_heldout.npy", shap_values)
    np.save(shap_dir / "shap_abs_all_folds_heldout.npy", shap_abs)
    np.save(shap_dir / "shap_signed_all_folds_heldout.npy", shap_signed)

    print(f"  Saved SHAP results to {shap_dir}")
    return {"shap_abs": shap_abs, "shap_signed": shap_signed, "masker": subj_masker}

In [ ]:
gcn_shap_results = {}
for subject in subjects:
    print("\n" + "=" * 70)
    print("SHAP - Subject:", subject)
    result = run_shap_for_subject(subject)
    if result is not None:
        gcn_shap_results[subject] = result

In [ ]:
for subject, result in gcn_shap_results.items():
    subj_dir = RESULTS_BASE_DIR / subject
    shap_dir = subj_dir / "shap"
    shap_figs = shap_dir / "figures"
    shap_figs.mkdir(parents=True, exist_ok=True)

    shap_abs = result["shap_abs"]
    subj_masker = result["masker"]

    for c in range(4):
        class_name = LABEL_TO_REGION[c]
        shap_img = subj_masker.inverse_transform(shap_abs[c].reshape(1, -1))
        texture = surf_utils.vol_to_surf(shap_img, fsaverage.pial_left)
        nonzero = texture[texture > 0]
        vmax = np.percentile(nonzero, 95) if len(nonzero) else 1.0
        thresh = np.percentile(nonzero, 75) if len(nonzero) else 0.0
        fig = plotting.plot_surf_stat_map(
            fsaverage.infl_left, stat_map=texture, hemi="left", view="lateral",
            cmap="hot", threshold=thresh, vmax=vmax, colorbar=True,
            symmetric_cmap=False, bg_on_data=True, alpha=0.7,
            title=f"GCN SHAP |importance|: {class_name}", cbar_tick_format="%.2e",
        )
        fig.savefig(str(shap_figs / f"shap_surf_{class_name}.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)

    TOP_PERCENTILE = 50
    winner_map = np.argmax(shap_abs, axis=0)
    max_shap = np.amax(shap_abs, axis=0)
    winner_vals = np.where(max_shap >= np.percentile(max_shap, TOP_PERCENTILE), winner_map + 1, 0).astype(float)
    winner_img = subj_masker.inverse_transform(winner_vals.reshape(1, -1))

    view = plotting.view_img_on_surf(
        winner_img, surf_mesh="fsaverage",
        title=f"{subject} - GCN SHAP Winner-Take-All (1=Finger, 2=Hand, 3=Forearm, 4=Arm)",
        symmetric_cmap=False, vmin=1, vmax=4, threshold=1,
        vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"},
    )
    view.save_as_html(shap_figs / "shap_winner_take_all.html")
    display(view)
    print(f"Saved SHAP surface plots for {subject}")